# Tokyo Real Estate: Group Assignment 2
**Data & Decisions** · Team members: _<add names>_

Inputs: `data/Chuo.xlsx`, `data/Minato.xlsx`, `data/Setagaya.xlsx`, `data/Shibuya.xlsx` (unchanged files from bCourses; SHA-256 checked against the downloads).
Scope: records whose `Type` is exactly **"Pre-owned Condominiums, etc."**. Unit of analysis: one reported 2022 transaction. Measure: price per m² = `Transaction-price(total)` ÷ `Area(m^2)`, in yen per m².

All random draws come from `SEED = 2022`, which was fixed before any sample was drawn and never changed. Running the notebook from top to bottom reproduces every table. The random number given to every record and the 600 selected `(Ward, No)` identifiers are also saved in `outputs/`.

| Output file | Contents |
|---|---|
| `part1_frame_counts.csv`, `part1_excluded_records.csv` | Record counts and every excluded record, with its reason |
| `part1_usable_frame.csv`, `part1_frame_means.csv` | The usable frame with ¥/m² for each record, and the ward means |
| `part1_verification_mean_vs_ratio.csv` | Verification of the averaging choice |
| `part2_random_numbers_all_records.csv` | The Uniform(0,1) number for every usable record, its rank, and whether it was selected |
| `part2_frozen_samples.csv`, `part2_sample_summary.csv` | The four frozen samples of 150, with summary statistics |
| `part3_confidence_intervals.csv`, `part3_welch_tests.csv`, `part3_overlap_vs_test.csv` | Part 3 results |

In [1]:
from pathlib import Path
from itertools import combinations
import numpy as np
import pandas as pd
from scipy import stats
import openpyxl

pd.set_option("display.float_format", "{:,.2f}".format)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

SEED = 2022             # fixed before any sample was drawn; never changed afterwards
N_SAMPLE = 150
ALPHA = 0.05
WARDS = ["Chuo", "Minato", "Setagaya", "Shibuya"]   # fixed processing order (alphabetical)
DATA = Path("data")
OUT = Path("outputs"); OUT.mkdir(exist_ok=True)

CONDO = "Pre-owned Condominiums, etc."
PRICE, AREA = "Transaction-price(total)", "Area(m^2)"
UNIT_PRICE = "Transaction-price(Unit price m^2)"

ModuleNotFoundError: No module named 'scipy'

## Part 1: Define the frame and verify the calculation

### 1a. Load the files and check the fields
Each file has a single sheet with the same 29 columns. `No` is a unique source identifier within each file, so `(Ward, No)` identifies a record. The `Type` labels are exact, with no spelling or whitespace variants. A looser "contains *condo*" match returns the same counts.

In [ ]:
raw = {w: pd.read_excel(DATA / f"{w}.xlsx", sheet_name=0) for w in WARDS}

for w, df in raw.items():
    assert df["No"].is_unique, f"{w}: No is not unique"
    assert (df["Type"] == CONDO).sum() == df["Type"].str.lower().str.contains("condo").sum()
    print(f"{w:9s} rows={len(df):5d}  columns={df.shape[1]}  No unique={df['No'].is_unique}")

print("\nType labels in each file (exact strings):")
print(pd.concat({w: df["Type"].value_counts() for w, df in raw.items()}, axis=1).fillna(0).astype(int))

Each record goes through these screens in order, and the first one it fails is recorded as its exclusion reason. Price and area are converted to numbers in a **separate copy**, so text values are never silently coerced.

1. `Type` is not "Pre-owned Condominiums, etc."
2. Total price or area is missing.
3. Total price or area is a bound (for example "2,000 m^2 or greater."). These are excluded, never recoded to an exact number.
4. Total price or area is non-numeric.
5. Total price or area is zero or negative.

In [ ]:
def screen(df, ward):
    """Flag each record's status for the usable frame, in priority order."""
    d = df.copy()
    d["Ward"] = ward
    d["record_id"] = ward + "-" + d["No"].astype(str)

    # Keep the original cell values for the checks; coerce a numeric copy separately.
    is_bound = lambda s: s.map(lambda v: isinstance(v, str) and ("greater" in v.lower() or "less" in v.lower()))
    d["price_num"] = pd.to_numeric(d[PRICE], errors="coerce")
    d["area_num"] = pd.to_numeric(d[AREA], errors="coerce")

    reason = pd.Series("", index=d.index)
    checks = [
        (d["Type"] != CONDO,                                      "not a pre-owned condominium (Type filter)"),
        (d[PRICE].isna() | d[AREA].isna(),                        "missing total price or area"),
        (is_bound(d[PRICE]) | is_bound(d[AREA]),                  "bounded value (e.g. '2,000 m^2 or greater')"),
        (d["price_num"].isna() | d["area_num"].isna(),            "non-numeric total price or area"),
        ((d["price_num"] <= 0) | (d["area_num"] <= 0),            "non-positive total price or area"),
    ]
    for mask, why in checks:                       # first failing check is the recorded reason
        reason = reason.mask((reason == "") & mask.fillna(False), why)
    d["exclusion_reason"] = reason
    d["usable"] = reason == ""
    return d

screened = pd.concat([screen(raw[w], w) for w in WARDS], ignore_index=True)

# Field-level checks on the condo rows, before any exclusion is applied
condo_rows = screened[screened["Type"] == CONDO]
qc = condo_rows.groupby("Ward").apply(lambda g: pd.Series({
    "condo rows": len(g),
    "price missing": g[PRICE].isna().sum(),
    "area missing": g[AREA].isna().sum(),
    "price non-numeric": (g["price_num"].isna() & g[PRICE].notna()).sum(),
    "area non-numeric": (g["area_num"].isna() & g[AREA].notna()).sum(),
    "price <= 0": (g["price_num"] <= 0).sum(),
    "area <= 0": (g["area_num"] <= 0).sum(),
    "unit-price field non-blank": g[UNIT_PRICE].notna().sum(),
    "min area (m2)": g["area_num"].min(),
    "max area (m2)": g["area_num"].max(),
}), include_groups=False).astype(int)
qc

None of the condo price or area cells is missing, non-numeric, non-positive or bounded. The existing `Transaction-price(Unit price m^2)` field is blank on all 2,502 condo rows (the last check column is 0 everywhere). We never use that field, so its blanks cannot enter the calculation as zeros.

Bounded values do occur in the files, but only in non-condo rows:

In [ ]:
# Every cell in the price and area columns of the full files that is text rather than a number
bounded = screened[screened[AREA].map(lambda v: isinstance(v, str)) | screened[PRICE].map(lambda v: isinstance(v, str))]
bounded[["Ward", "No", "Type", PRICE, AREA]]

These three "2,000 m^2 or greater." areas belong to *Residential Land (Land Only)* records. The Type filter removes them, and they are never converted to 2,000. They are also why pandas reads the `Area(m^2)` column as text in the Chuo and Setagaya files. Every condo area in those files is still a true integer.

### 1b. Record counts and exclusions

In [ ]:
counts = screened.groupby("Ward").agg(
    original_records=("No", "size"),
    condominium_records=("Type", lambda s: (s == CONDO).sum()),
    usable_records=("usable", "sum"),
)
counts["excluded_not_condo"] = counts["original_records"] - counts["condominium_records"]
counts["excluded_condo_failed_checks"] = counts["condominium_records"] - counts["usable_records"]
counts.loc["Total"] = counts.sum()
counts.to_csv(OUT / "part1_frame_counts.csv")
counts

In [ ]:
reasons = (screened[~screened["usable"]]
           .groupby(["Ward", "exclusion_reason", "Type"]).size().rename("records").to_frame())
excluded = screened.loc[~screened["usable"], ["Ward", "No", "record_id", "Type", PRICE, AREA, "exclusion_reason"]]
excluded.to_csv(OUT / "part1_excluded_records.csv", index=False)
reasons

**1b answer.**

| Ward | Original records | Condominium records | Usable | Excluded: not a condo (land + building / land only) | Excluded: condo failed a check |
|---|---:|---:|---:|---:|---:|
| Chuo | 665 | 603 | 603 | 62 (50 / 12) | 0 |
| Minato | 833 | 727 | 727 | 106 (79 / 27) | 0 |
| Setagaya | 1,488 | 746 | 746 | 742 (515 / 227) | 0 |
| Shibuya | 538 | 426 | 426 | 112 (74 / 38) | 0 |
| **Total** | **3,524** | **2,502** | **2,502** | **1,022** | **0** |

The only exclusion reason is the Type filter. No condominium record fails the missing, bounded, non-numeric or non-positive checks. The full list of excluded records is in `outputs/part1_excluded_records.csv`.

### 1c. Full-frame ward means of price per m²

In [ ]:
frame = screened[screened["usable"]].copy()
frame["ppsqm"] = frame["price_num"] / frame["area_num"]          # yen per square metre, per transaction
frame = frame.sort_values(["Ward", "No"]).reset_index(drop=True)

keep_cols = ["Ward", "No", "record_id", "Area", "Nearest stationÅFName", "Transaction period",
             "Layout", "Year of construction", PRICE, AREA, "ppsqm"]
frame[keep_cols].to_csv(OUT / "part1_usable_frame.csv", index=False)

frame_stats = frame.groupby("Ward")["ppsqm"].agg(
    N="size", mean="mean", sd=lambda s: s.std(ddof=1), median="median", min="min", max="max")
frame_stats.to_csv(OUT / "part1_frame_means.csv")
frame_stats

In [ ]:
# Spot check: three records that are easy to verify by hand
frame.loc[[0, 1, 2], ["record_id", PRICE, AREA, "ppsqm"]]

**1c answer.** Arithmetic mean of transaction-level price per m², over each full usable frame:

| Ward | N | Mean ¥/m² | Median ¥/m² | SD ¥/m² |
|---|---:|---:|---:|---:|
| Minato | 727 | **1,545,907** | 1,425,000 | 600,466 |
| Shibuya | 426 | **1,339,364** | 1,250,000 | 482,990 |
| Chuo | 603 | **1,256,459** | 1,240,000 | 585,865 |
| Setagaya | 746 | **974,634** | 960,000 | 295,136 |

Spot check: Chuo No 1 sold for ¥79,000,000 with 55 m², and 79,000,000 / 55 = ¥1,436,364 per m², which matches the table.

### 1d. Verification of one consequential analytical choice

**The choice:** how to average. We take the arithmetic mean of each transaction's own ratio, (1/N) Σ(priceᵢ / areaᵢ), so every sale counts equally. The tempting shortcut is total yen divided by total m², Σpriceᵢ / Σareaᵢ. That is an area-weighted average, so large units count for more. **The check:** compute both for every ward and compare the levels and the ranking.

In [ ]:
# Verification 1: the consequential choice, mean of per-transaction ratios vs ratio of totals.
ratio_of_totals = frame.groupby("Ward").apply(lambda g: g["price_num"].sum() / g["area_num"].sum(), include_groups=False)
verif = pd.DataFrame({
    "mean of per-transaction ¥/m² (used)": frame_stats["mean"],
    "Σprice / Σarea (not used)": ratio_of_totals,
})
verif["difference"] = verif.iloc[:, 1] - verif.iloc[:, 0]
verif["% difference"] = 100 * verif["difference"] / verif.iloc[:, 0]
verif["rank (used)"] = verif.iloc[:, 0].rank(ascending=False).astype(int)
verif["rank (not used)"] = verif.iloc[:, 1].rank(ascending=False).astype(int)
# Correlation between size and ¥/m² explains the direction of the gap
verif["corr(area, ¥/m²)"] = frame.groupby("Ward").apply(lambda g: g["area_num"].corr(g["ppsqm"]), include_groups=False)
verif.to_csv(OUT / "part1_verification_mean_vs_ratio.csv")
verif

We also recomputed the ward means a second, independent way. The script below reads the workbook cells directly with `openpyxl`, so it does not depend on pandas' type inference, and it must reproduce the same N and mean:

In [ ]:
# Verification 2: independent recomputation straight from the workbook cells with openpyxl,
# matching columns by their header names.
def recompute_with_openpyxl(path):
    ws = openpyxl.load_workbook(path, read_only=True, data_only=True).worksheets[0]
    rows = ws.iter_rows(values_only=True)
    header = next(rows)
    i_type, i_price, i_area = header.index("Type"), header.index(PRICE), header.index(AREA)
    vals = [r[i_price] / r[i_area] for r in rows
            if r[i_type] == CONDO and isinstance(r[i_price], (int, float)) and isinstance(r[i_area], (int, float))
            and r[i_price] > 0 and r[i_area] > 0]
    return len(vals), sum(vals) / len(vals)

indep = pd.DataFrame([recompute_with_openpyxl(DATA / f"{w}.xlsx") for w in WARDS],
                     index=WARDS, columns=["N (openpyxl)", "mean ¥/m² (openpyxl)"])
indep["mean ¥/m² (pandas)"] = frame_stats["mean"]
indep["abs diff"] = (indep["mean ¥/m² (openpyxl)"] - indep["mean ¥/m² (pandas)"]).abs()
assert (indep["N (openpyxl)"] == frame_stats["N"]).all() and (indep["abs diff"] < 1e-6).all()
indep

Two further robustness checks: rows that repeat every field except `No`, and the most extreme record in the frame.

In [ ]:
# Verification 3: rows identical in every field except No. We keep them, because anonymised and rounded
# data can make two distinct sales in one building look identical. This shows the mean is not sensitive to that choice.
dup_cols = [c for c in raw["Chuo"].columns if c != "No"]
frame["dup_except_No"] = frame.duplicated(subset=["Ward"] + dup_cols, keep="first")
dup_check = pd.DataFrame({
    "repeat rows": frame.groupby("Ward")["dup_except_No"].sum(),
    "mean, all rows kept (used)": frame_stats["mean"],
    "mean, repeats dropped": frame[~frame["dup_except_No"]].groupby("Ward")["ppsqm"].mean(),
})
dup_check["% change"] = 100 * (dup_check.iloc[:, 2] / dup_check.iloc[:, 1] - 1)
dup_check

In [ ]:
# Verification 4: the most extreme ¥/m² in the frame (kept; shown for transparency)
top = frame.sort_values("ppsqm", ascending=False).groupby("Ward").head(2).sort_values(["Ward", "ppsqm"], ascending=[True, False])
display(top[["record_id", "Area", "Layout", "Year of construction", "Transaction period", PRICE, AREA, "ppsqm"]].astype({"Year of construction": "Int64"}))
chuo = frame[frame["Ward"] == "Chuo"]
print(f"Chuo frame mean with Chuo-40:    ¥{chuo['ppsqm'].mean():,.0f}/m²")
print(f"Chuo frame mean without Chuo-40: ¥{chuo.loc[chuo['record_id'] != 'Chuo-40', 'ppsqm'].mean():,.0f}/m² (sensitivity only)")

**1d answer: verification.**

- **The choice changes the levels but not the ranking.** Total-price ÷ total-area would raise every ward's figure: Setagaya by 1.4%, Chuo by 3.6%, Shibuya by 10.3% and Minato by 12.0%. The reason is that within each ward, larger units also sell for more *per m²* (corr(area, ¥/m²) is 0.09 to 0.50, strongest in Minato and Shibuya). The area-weighted version therefore tilts toward large luxury units and no longer describes the typical transaction. The ward ranking is the same either way: **Minato > Shibuya > Chuo > Setagaya**. Some gaps do move. For example, Shibuya − Chuo is ¥82,905 using the per-transaction mean but ¥174,930 using the ratio of totals. So the choice does not change our ranking conclusion, but it would materially change any reported level or gap. We use the per-transaction mean, as the assignment requires.
- **The calculation reproduces exactly.** The openpyxl recomputation matches pandas on N and on the mean to within ¥0.000001 in all four wards.
- **The repeated rows don't matter.** 2, 2, 1 and 2 condo rows (by ward) repeat every field except `No`. We keep them, because rounded, anonymised records of separate sales in the same building can look identical. Dropping them moves each mean by at most 0.11%.
- **One extreme record is kept.** Chuo-40 is listed at ¥900M for a 65 m², 3LDK unit in Kachidoki built in 2007, which is ¥13.8M/m². That is 5.5 times the next-highest Chuo sale (¥2.53M/m²). It is numeric and positive, so we keep it, as the assignment instructs. Without it the Chuo mean would be ¥1,235,546 (−1.7%), and the ranking would not change. It matters much more in the sample (Parts 2 and 3).

### 1e. Whom the ward averages describe, and a limit
The averages describe **2022 pre-owned condominium sales in each ward that were reported in MLIT's quarterly questionnaire survey and included in these files**. Each reported transaction counts equally. They are not the average value of all condominium stock, they do not cover new-build condos, and they are not area-weighted.

**Limit to generalizing:** the frame is the set of survey responses, not a census of sales. If response rates vary with the kind of sale (for example, luxury sales, distressed sales or related-party deals answering less often), the frame mean can differ systematically from the true market mean. Random sampling from these files cannot fix that. The averages also mix very different properties (sizes, ages, distances to stations). So a gap between wards partly reflects a different mix of property, not only a different price for the same property.

## Part 2: Draw reproducible samples (n = 150 per ward)

**Method.** One generator, `np.random.default_rng(2022)`, gives every usable record a Uniform(0,1) number. Records are processed ward by ward in alphabetical order, and within each ward in ascending `No` order. Within each ward, the **150 records with the smallest numbers** are selected. This is a simple random sample without replacement, because every set of 150 records is equally likely. Every random number and its rank is saved, so the selection can be re-checked in Excel by sorting `outputs/part2_random_numbers_all_records.csv`. The samples were drawn once and are frozen. Part 3 reads them back from `outputs/part2_frozen_samples.csv`, and nothing was redrawn.

In [ ]:
rng = np.random.default_rng(SEED)
draws = []
for w in WARDS:
    g = frame[frame["Ward"] == w].sort_values("No").copy()
    g["U"] = rng.random(len(g))
    g["U_rank"] = g["U"].rank(method="first").astype(int)
    g["selected"] = g["U_rank"] <= N_SAMPLE
    draws.append(g)
draws = pd.concat(draws, ignore_index=True)

draws[["Ward", "No", "record_id", "U", "U_rank", "selected", PRICE, AREA, "ppsqm"]].to_csv(
    OUT / "part2_random_numbers_all_records.csv", index=False)

sample = draws[draws["selected"]].sort_values(["Ward", "No"]).reset_index(drop=True)
sample[["Ward", "No", "record_id", "U", PRICE, AREA, "ppsqm"]].to_csv(OUT / "part2_frozen_samples.csv", index=False)

assert (sample.groupby("Ward").size() == N_SAMPLE).all()
assert not sample.duplicated(["Ward", "No"]).any()          # without replacement

**Selected identifiers.** These are the source `No` values for each ward, 150 per ward, so a record is identified by `(Ward, No)`.

In [ ]:
ids = sample.groupby("Ward")["No"].apply(lambda s: ", ".join(map(str, sorted(s))))     # selected identifiers

In [ ]:
samp_stats = sample.groupby("Ward")["ppsqm"].agg(n="size", mean="mean", sd=lambda s: s.std(ddof=1))
part2 = samp_stats.copy()
part2["frame mean (N)"] = frame_stats["mean"]
part2["N"] = frame_stats["N"]
part2["sample − frame"] = part2["mean"] - part2["frame mean (N)"]
part2["% gap"] = 100 * part2["sample − frame"] / part2["frame mean (N)"]
part2["gap in SE units"] = part2["sample − frame"] / (part2["sd"] / np.sqrt(part2["n"]))
part2.to_csv(OUT / "part2_sample_summary.csv")     # save sample statistics output to csv

**Part 2 answer:** *For each sample, report n, the mean price per square meter, and the sample standard deviation. Compare
the sample mean with the full-frame mean from Part 1.*

| Ward | n | Sample mean ¥/m² | Sample SD ¥/m² | Frame mean ¥/m² (N) | Sample − frame | % gap | Gap ÷ SE |
|---|---:|---:|---:|---:|---:|---:|---:|
| Chuo | 150 | 1,332,712 | 1,061,122 | 1,256,459 (603) | +76,253 | +6.1% | +0.88 |
| Minato | 150 | 1,500,346 | 519,744 | 1,545,907 (727) | −45,561 | −3.0% | −1.07 |
| Setagaya | 150 | 976,456 | 320,063 | 974,634 (746) | +1,822 | +0.2% | +0.07 |
| Shibuya | 150 | 1,359,885 | 491,412 | 1,339,364 (426) | +20,521 | +1.5% | +0.51 |

**Why a correctly drawn sample need not match its frame exactly:** A sample is a representative subset of the full frame, in this case 600 items out of ~2,500. All items have an equal chance of being chosen, so the sample is correctly drawn, but we still see randomness and sampling error in the results because we aren't seeing the full frame. The sample mean is a random variable centered on the frame mean, and it spreads around the frame mean by about one standard error, s/√n. With our random sample here, all four gaps are within about one standard error, which is about the size of chance variation expected.

## Part 3: Quantify uncertainty and compare wards

### 3a. 95% confidence interval for each ward's mean
Course formula: x̄ ± t₀.₀₂₅,ₙ₋₁ · s/√n, with df = n − 1 = 149, so t₀.₀₂₅,₁₄₉ = 1.976. No finite-population correction is applied.

In [ ]:
# Part 3 reads the frozen samples back from disk, so it uses exactly the saved selection.
frozen = pd.read_csv(OUT / "part2_frozen_samples.csv")
assert set(zip(frozen["Ward"], frozen["No"])) == set(zip(sample["Ward"], sample["No"]))

s = frozen.groupby("Ward")["ppsqm"].agg(n="size", mean="mean", sd=lambda x: x.std(ddof=1))
ci = s.copy()
ci["se = s/√n"] = ci["sd"] / np.sqrt(ci["n"])
ci["df"] = ci["n"] - 1
ci["t crit (0.025, df)"] = stats.t.ppf(1 - ALPHA / 2, ci["df"])
ci["margin"] = ci["t crit (0.025, df)"] * ci["se = s/√n"]
ci["lower 95%"] = ci["mean"] - ci["margin"]
ci["upper 95%"] = ci["mean"] + ci["margin"]
ci["frame mean"] = frame_stats["mean"]
ci["covers frame mean?"] = (ci["lower 95%"] <= ci["frame mean"]) & (ci["frame mean"] <= ci["upper 95%"])
ci.to_csv(OUT / "part3_confidence_intervals.csv")
with pd.option_context("display.float_format", "{:,.3f}".format):
    display(ci[["n", "df", "t crit (0.025, df)"]])
ci

**3a answer.**

| Ward | Mean ¥/m² | SE = s/√150 | t crit (df 149) | Margin | 95% CI lower | 95% CI upper |
|---|---:|---:|---:|---:|---:|---:|
| Chuo | 1,332,712 | 86,640 | 1.976 | 171,202 | 1,161,510 | 1,503,914 |
| Minato | 1,500,346 | 42,437 | 1.976 | 83,856 | 1,416,490 | 1,584,202 |
| Setagaya | 976,456 | 26,133 | 1.976 | 51,639 | 924,817 | 1,028,095 |
| Shibuya | 1,359,885 | 40,124 | 1.976 | 79,285 | 1,280,600 | 1,439,170 |

All four intervals contain their known frame means, which we can check only because this exercise has the full frame. Chuo's interval is about twice as wide as Minato's or Shibuya's, almost entirely because of Chuo-40's effect on its SD.

### 3b. Welch two-sample tests for all six pairs
H₀: μ_A − μ_B = 0 vs Hₐ: μ_A − μ_B ≠ 0, at α = 0.05. **Subtraction order: A − B, where A is the alphabetically earlier ward** (Chuo − Minato, Chuo − Setagaya, and so on). The SE is unpooled, se = √(s_A²/n_A + s_B²/n_B), and the reference t distribution uses Welch–Satterthwaite degrees of freedom. Each result is cross-checked against `scipy.stats.ttest_ind(equal_var=False)`. A large-sample normal p-value is shown for comparison.

In [ ]:
rows = []
for a, b in combinations(WARDS, 2):          # difference is always (first ward − second ward), alphabetical
    xa, xb = frozen.loc[frozen["Ward"] == a, "ppsqm"], frozen.loc[frozen["Ward"] == b, "ppsqm"]
    na, nb = len(xa), len(xb)
    va, vb = xa.var(ddof=1) / na, xb.var(ddof=1) / nb
    diff = xa.mean() - xb.mean()
    se = np.sqrt(va + vb)
    t = diff / se
    df = (va + vb) ** 2 / (va ** 2 / (na - 1) + vb ** 2 / (nb - 1))    # Welch–Satterthwaite
    tcrit = stats.t.ppf(1 - ALPHA / 2, df)
    p = 2 * stats.t.sf(abs(t), df)
    p_z = 2 * stats.norm.sf(abs(t))                                     # large-sample normal check
    chk = stats.ttest_ind(xa, xb, equal_var=False)                      # library cross-check
    assert np.isclose(chk.statistic, t) and np.isclose(chk.pvalue, p)
    rows.append({
        "pair (A − B)": f"{a} − {b}",
        "mean A": xa.mean(), "mean B": xb.mean(),
        "difference A − B": diff, "se(diff)": se, "t": t, "Welch df": df, "t crit": tcrit,
        "p-value": p, "p (normal approx.)": p_z,
        "95% CI diff lower": diff - tcrit * se, "95% CI diff upper": diff + tcrit * se,
        "frame diff A − B": frame_stats.loc[a, "mean"] - frame_stats.loc[b, "mean"],
        "reject H0 at 5%?": p < ALPHA,
    })
tests = pd.DataFrame(rows).set_index("pair (A − B)")
tests.to_csv(OUT / "part3_welch_tests.csv")
with pd.option_context("display.float_format", "{:,.4f}".format):
    display(tests[["t", "Welch df", "t crit", "p-value", "p (normal approx.)", "reject H0 at 5%?"]])
tests[["mean A", "mean B", "difference A − B", "se(diff)", "95% CI diff lower", "95% CI diff upper", "frame diff A − B"]]

**3b answer.** Differences are A − B in ¥/m².

| Pair (A − B) | Difference | se(diff) | t | Welch df | p-value | 95% CI for difference | Conclusion at 5% |
|---|---:|---:|---:|---:|---:|---|---|
| Chuo − Minato | −167,634 | 96,475 | −1.74 | 216.6 | 0.084 | [−357,784, +22,516] | Fail to reject |
| Chuo − Setagaya | +356,256 | 90,496 | 3.94 | 175.9 | 0.00012 | [+177,659, +534,853] | **Reject**: Chuo higher |
| Chuo − Shibuya | −27,173 | 95,480 | −0.28 | 210.1 | 0.776 | [−215,395, +161,048] | Fail to reject |
| Minato − Setagaya | +523,890 | 49,838 | 10.51 | 247.8 | < 0.001 | [+425,729, +622,050] | **Reject**: Minato higher |
| Minato − Shibuya | +140,460 | 58,402 | 2.41 | 297.1 | 0.017 | [+25,526, +255,395] | **Reject**: Minato higher |
| Setagaya − Shibuya | −383,429 | 47,884 | −8.01 | 256.1 | < 0.001 | [−477,725, −289,133] | **Reject**: Shibuya higher |

The critical values are t₀.₀₂₅ ≈ 1.968 to 1.974 at these degrees of freedom. The large-sample normal p-values lead to the same six conclusions.

### 3c. Tests vs the four separate confidence intervals

In [ ]:
ov = []
for a, b in combinations(WARDS, 2):
    lo = max(ci.loc[a, "lower 95%"], ci.loc[b, "lower 95%"])
    hi = min(ci.loc[a, "upper 95%"], ci.loc[b, "upper 95%"])
    sea, seb = ci.loc[a, "se = s/√n"], ci.loc[b, "se = s/√n"]
    ov.append({
        "pair (A − B)": f"{a} − {b}",
        "individual CIs overlap?": lo <= hi,
        "overlap width (¥/m²)": max(0.0, hi - lo),
        "se_A + se_B": sea + seb,
        "√(se_A² + se_B²) = se(diff)": np.sqrt(sea ** 2 + seb ** 2),
        "Welch test rejects?": tests.loc[f"{a} − {b}", "reject H0 at 5%?"],
    })
overlap = pd.DataFrame(ov).set_index("pair (A − B)")
overlap.to_csv(OUT / "part3_overlap_vs_test.csv")
overlap

**3c answer.** For three pairs (Chuo − Setagaya, Minato − Setagaya and Setagaya − Shibuya) the two individual intervals do not overlap, and the test rejects. For Chuo − Minato and Chuo − Shibuya they overlap, and the test does not reject. **Minato − Shibuya is the instructive case.** Its intervals overlap by ¥22,680, yet the Welch test rejects equality (p = 0.017).

**Why this happens.** Two separate 95% intervals fail to overlap only when the gap in means exceeds roughly t·(se_A + se_B). The test asks a different question: is the gap large relative to the uncertainty *of the difference*, se(diff) = √(se_A² + se_B²)? Because √(se_A² + se_B²) < se_A + se_B, the test needs a smaller gap. For Minato − Shibuya:
- Non-overlap would need the gap to exceed about 1.976 × (42,437 + 40,124) = ¥163,141.
- The test needs it to exceed only 1.968 × 58,402 = ¥114,935.
- The observed gap of ¥140,460 lies between the two thresholds, so the intervals overlap even though the test rejects.

The right tool for comparing two wards is the **interval for the difference**: [+¥25,526, +¥255,395], which excludes zero. Overlap of the separate intervals is a conservative visual guide, not a formal test. Non-overlap does imply a rejection, but overlap does not imply failure to reject.

### 3d. Conclusions, with appropriate caution

In [ ]:
# Six tests at 5%: chance of at least one false rejection if every null were true (independence as a rough guide)
print(f"1 − 0.95^6 = {1 - 0.95**6:.3f}   expected false rejections = 6 × 0.05 = {6*0.05:.2f}")
print("Highest sample mean:", s["mean"].idxmax(), f"(¥{s['mean'].max():,.0f}/m²)")
print("Highest frame mean: ", frame_stats["mean"].idxmax(), f"(¥{frame_stats['mean'].max():,.0f}/m²)")

In [ ]:
# Sensitivity only, NOT the reported result: the two Chuo tests if the single extreme record Chuo-40 is set aside.
chuo_wo = frozen.loc[(frozen["Ward"] == "Chuo") & (frozen["No"] != 40), "ppsqm"]
sens = []
for b in ["Minato", "Shibuya"]:
    r = stats.ttest_ind(chuo_wo, frozen.loc[frozen["Ward"] == b, "ppsqm"], equal_var=False)
    sens.append({"pair": f"Chuo − {b}", "reported p (Chuo-40 kept)": tests.loc[f"Chuo − {b}", "p-value"],
                 "difference without Chuo-40": chuo_wo.mean() - frozen.loc[frozen["Ward"] == b, "ppsqm"].mean(),
                 "t without Chuo-40": r.statistic, "p without Chuo-40": r.pvalue})
with pd.option_context("display.float_format", "{:,.4f}".format):
    display(pd.DataFrame(sens).set_index("pair"))

**3d answer.**

- **Highest sample mean:** Minato, at ¥1,500,346/m² (95% CI ¥1.42M to ¥1.58M). Minato also has the highest frame mean.
- **Pairs where the samples support a difference (5% level):** Setagaya is lower than each of the other three wards (all p < 0.001), and Minato is higher than Shibuya (p = 0.017).
- **Pairs where they do not:** Chuo vs Minato (p = 0.084) and Chuo vs Shibuya (p = 0.776).

**Failing to reject is not proof that prices are equal.** The 95% CI for Chuo − Minato, [−¥357,784, +¥22,516], includes zero, but it also includes gaps of more than ¥300k/m². The data cannot rule out a large difference. In this exercise we can see that the frame gap really is −¥289,447 (Minato higher), so this test is a missed difference (a Type II error), not evidence of equality. The main reason is Chuo-40. That one record inflates Chuo's SE. If it is set aside purely as a sensitivity check, Chuo − Minato becomes p < 0.001 and Chuo − Shibuya becomes p = 0.015. Our conclusions about Chuo are therefore fragile, and before relying on them we would verify that transaction with MLIT or a listing source. We do not drop it from the reported analysis, because it is a valid, numeric, positive record.

**Six tests create room for chance findings.** Each test has a 5% false-rejection rate when its null hypothesis is true. If all six nulls were true and the tests were independent, the chance of at least one false "significant" result would be 1 − 0.95⁶ ≈ 26%. The tests share wards, so they are not independent, but the point stands. The three Setagaya rejections have p < 0.001 and are very unlikely to be chance. Minato − Shibuya, at p = 0.017, is the finding most exposed to this concern.

**No definitive ranking.** The samples suggest Minato is the most expensive per m² and Setagaya the least, with Chuo and Shibuya not separable from these data. This is not a ranking of Tokyo's whole market. It covers four wards, one year, one property type, and only the survey-reported transactions.

### 3e. How to interpret these intervals and tests
The frame means are known here, so the inference is practice for what a 150-sale sample would reveal. The intervals and tests capture **only random sampling error from these files**. They do not account for:
- **Survey nonresponse or coverage gaps.** Transactions that were never reported are outside the frame.
- **Differences in property mix.** Ward gaps compare different sizes, ages, layouts and locations, not like-for-like units.
- **Rounding and anonymisation in the source.**

A statistically significant difference here means the *reported* condo sales in two wards differ in average ¥/m² by more than chance would explain. It does not mean that an identical apartment costs that much more in one ward than the other.